In [ ]:
import tensorflow as tf 
import keras
from tensorflow.python.data import Dataset
import numpy as np
from argparse import ArgumentParser

# # Kết nối với TPU
# resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
# # kết nối
# tf.config.experimental_connect_to_cluster(resolver)
# # khởi tạo 
# tf.tpu.experimental.initialize_tpu_system(resolver)
# # xây dựng môi trường phân tán 
# strategy = tf.distribute.experimental.TPUStrategy(resolver)

In [ ]:
class PATCHES(keras.layers.Layer):
    def __init__(self, patch_size):
        super(PATCHES, self).__init__()
        self.patch_size = patch_size 
    
    def call(self, images):
        batch_size = tf.shape(images)[0] # batch_size from size of tensor images 
        # exxtract patches from image 
        patches = tf.image.extract_patches(
            images = images, 
            sizes = [1, self.patch_size, self.patch_size, 1], 
            strides = [1, self.patch_size, self.patch_size, 1], 
            rates = [1, 1, 1, 1],
            padding='VALID',
        )
        dim = patches.shape[-1]
        
        # return patches tensor shape [bbatch_size, num_patches, dim]
        patches = tf.reshape(patches, (batch_size, -1, dim))
        return patches 
    


    
# Create class Patches embedidng 
class PatchesEmbedding(keras.layers.Layer):
    def __init__(self, patch_size, image_size, projection_dim):
        super(PatchesEmbedding, self).__init__()
        
        # S = self.num_paches from image size 
        self.num_patches =(image_size // patch_size) ** 2
        
        # create cls_tokens  for lasr mlp network 
        self.cls_token = self.add_weight(
            name="cls_token",  # Đặt tên cho weight
            shape=(1, 1, projection_dim),  # Đảm bảo không có giá trị trùng lặp ở đây
            initializer=tf.keras.initializers.RandomNormal(),  # Sử dụng RandomNormal
            dtype=tf.float32,
        )
        self.patches = PATCHES(patch_size)
        
        # linear projection dim 
        self.projection = keras.layers.Dense(units=projection_dim)
        
        # ignore absolute position embedding using Rop EMBEDDING 
        self.position_embedding = self.add_weight(
            name="position_embeddings",
            shape=(self.num_patches, projection_dim),
            initializer=tf.keras.initializers.RandomNormal(),
            dtype=tf.float32
        )
    
    def call(self, images):
        
        # Get ppatchesfrom image  shape [batch_size, num_patches, projection dim]
        patch = self.patches(images)
        
        # encoded_patches shape: (..., S, D)
        encoded_patches = self.projection(patch)
        
        batch_size = tf.shape(images)[0]
        
        hidden_size = tf.shape(encoded_patches)[-1]
        
        # cls_broadcasted shape: (..., 1, D)
        cls_broadcasted = tf.cast(
            tf.broadcast_to(self.cls_token, [batch_size, 1, hidden_size]),
            dtype=images.dtype,
        )

        # # encoded_patches shape: (..., S + 1, D)
        # encoded_patches = tf.concat([cls_broadcasted, encoded_patches], axis=1)

        # # encoded_patches shape: (..., S + 1, D)
        # encoded_patches = encoded_patches # + self.position_embedding

         # encoded_patches shape: (..., S, D)
        encoded_patches = encoded_patches + self.position_embedding

        # encoded_patches shape: (..., S + 1, D)
        encoded_patches = tf.concat([cls_broadcasted, encoded_patches], axis=1)

        return encoded_patches

In [ ]:
class MLP(keras.layers.Layer):
    def __init__(self, hidden_layers, dropout=0.1, activation='gelu'):
        
        super(MLP, self).__init__()
        layers = []
        for num_units in hidden_layers: 
            layers.extend([
                keras.layers.Dense(num_units, activation=activation),
                keras.layers.Dropout(dropout)
            ])

        self.mlp = keras.Sequential(layers)
        
    def call(self, inputs, *args, **kwargs):
        outputs = self.mlp(inputs, *args, **kwargs)
        return outputs 


class RMSNorm(keras.layers.Layer):
    def __init__(self, units, epsilon=1e-12, **kwargs):
        super(RMSNorm, self).__init__(**kwargs)
        self.units = units
        self.epsilon = epsilon
        self.gamma = None

    def build(self, input_shape):
        self.gamma = self.add_weight(
            name="gamma",
            shape=(self.units,),
            initializer="ones",
            trainable=True,
        )

    def _norm(self, x):
        # Calculate the mean and variance of the input tensor
        mean = tf.reduce_mean(x, axis=-1, keepdims=True)
        variance = tf.reduce_mean(tf.square(x - mean), axis=-1, keepdims=True)
        # Normalize the input tensor
        return (x - mean) / tf.sqrt(variance + self.epsilon)

    def call(self, x):
        # Apply the normalization and scaling
        return self.gamma * self._norm(tf.cast(x, tf.float32)) # Cast the input tensor to float32
        

def precompute_theta_pos_frequenceis(head_dim, num_patches, theta = 10000.0):
    # create theta tensor  shape = [head_dim / 2]
    theta_number = tf.range(0, head_dim, 2, dtype=tf.float32)
    
    #Tính toán 1 góc quay theta 
    theta_number = 1.0 * (theta ** (theta_number / head_dim))
    # khởi tạo một tensor m shape = [num_patches] tensor này được sử dụng để mở rộng tensor freqs 
    m = tf.range(num_patches, dtype= tf.float32)
    
    # Shape: (num_patches) outer_product (Head_Dim / 2) -> (Seq_Len, Head_Dim / 2)
    freqs = tf.tensordot(m, theta, axes=0)
    
    #compute complex numbers in polar form 
    # shape = [num_patches, head_dim / 2]
    freqs_complex = tf.complex(tf.ones_like(freqs), freqs)
    return freqs_complex 
    

def apply_rotary_embedding(x, freqs_complex):
    # x shape [batch_size, num_heads, num_patches + 1, head_dim]
    x_first = x[:, :, :1, :]  # Kỳ vọng [batch_size, num_heads, 1, head_dim] 
    
    x_last = x[:, :, 1:, :]  # [batch_size, num_heads, num_patches, head_dim]
    x_complex = tf.complex(x_last[..., ::2], x_last[..., 1::2])  # shape [batch_size, num_heads, num_patches, head_dim / 2]

    # shape [seq_length, head_dim / 2] -> shape [1, seq_length, 1, head_dim / 2]
    freqs_complex = tf.expand_dims(tf.expand_dims(freqs_complex, axis=0), axis=2) 

    # Tính toán tensor x_rotated
    x_rotated = x_complex * freqs_complex[:, :1, :]  # Tránh lấy phần tử ngoài số lượng num_patches
    
    # Chuyển đổi số phức trở lại thành số thực và nối lại với phần đầu (CLS token)
    # Nối phần thực và phần ảo của x_rotated trên chiều cuối cùng (axis=-1)
    x_rotated_real_imag = tf.concat([tf.math.real(x_rotated), tf.math.imag(x_rotated)], axis=-1)
    
    # Nối x_first (có shape [batch_size, num_head, 1, head_dim]) với x_rotated_real_imag (có shape [batch_size, num_head, num_patches + 1, head_dim])
    # Chú ý rằng cần phải đảm bảo tensor x_first có thể nối với x_rotated_real_imag đúng chiều.
    x_out = tf.concat([x_first, x_rotated_real_imag], axis=2)
    
    # print(f"x_out shape: {x_out.shape}")
    return x_out


In [ ]:
class MultiHeadAttention(keras.layers.Layer):
    def __init__(self, num_heads, embedding_dim): 
        super(MultiHeadAttention, self).__init__()
        
        self.num_heads = num_heads
        self.embed_dim = embedding_dim 
        self.head_dim = embedding_dim // num_heads
        
        self.cal = (embedding_dim // num_heads) ** -3.14
        
        self.wq = keras.layers.Dense(embedding_dim) # kernel_initializer="glorot_uniform")
        self.wk = keras.layers.Dense(embedding_dim) # kernel_initializer="glorot_uniform")
        self.wv = keras.layers.Dense(embedding_dim) # kernel_initializer="glorot_uniform")
        
        self.w0 = keras.layers.Dense(embedding_dim)
        
        self.dropout = keras.layers.Dropout(rate=0.1)
        
    def split_head(self, x, batch_size):
        # reshape tensor shape [batch_size, num_patches + 1, embed_dim] -> tensor shape [ batch_size, num_patches + 1, num_head, head_dim]
        x = tf.reshape(x, (batch_size, tf.shape(x)[1], self.num_heads, self.head_dim))
        
        # Transpose tensor shape [ batch_size, num_head, num_patches + 1, head_dim]
        x = tf.transpose(x, perm=[0, 2, 1, 3])
        return x 
    
    def call(self, inputs): 
        # inputs shape [batch_size, seq_length, embed_dim]
        batch_size = tf.shape(inputs)[0]
        num_patches = tf.shape(inputs)[1]
        
        # cretae q, k, v vector 
        Q = self.wq(inputs)
        K = self.wk(inputs)
        V = self.wv(inputs)
        
        # K = K * self.cal  # user K
          
        # reshape tensor 
        Q = self.split_head(Q, batch_size)
        K = self.split_head(K, batch_size)
        V = self.split_head(V, batch_size)
        
        # Apply rotary embedding 
        freqs_complex = precompute_theta_pos_frequenceis(self.head_dim, num_patches)
        
        Q = apply_rotary_embedding(Q, freqs_complex)
        K = apply_rotary_embedding(K, freqs_complex)
        
        # Tính toán attention q.K 
        # shape [batch_size, num_head, sequence_length, sequence_length]
        # shape Q = [batch_size, num_head, seq_length, head_dim] * shape [batch_size, num_head, head_dim, sequence_length]
        attention_score = tf.matmul(Q, K, transpose_b=True)
        # attention / dk 
        d = tf.cast(self.head_dim, tf.float32)
        score = attention_score / tf.math.sqrt(d)
        
        
        # bỏ qua việc áp dụng attention mask  áp dụn trực tiếp hàm softmax 
        attention_weights = tf.nn.softmax(score, axis=-1)
        attention_weights = self.dropout(attention_weights)

        # nhân vector attention với v
        # shape tensor = batch_size, num_head, seq_length, seq_length * batch_size, num_head, seq_length, embed_dim - >
        # shape = batch_size, num_head, seq_length, embed_dim
        attention_ = tf.matmul(attention_weights, V)
        # transpose shape = batch_size, seqence_length, num_head, head_Dim
        attention_ = tf.transpose(attention_, perm=[0, 2, 1, 3])
        # reshape tensor -> shape [batch_size, seq_length, embed_dim]
        attention_ = tf.reshape(attention_, (batch_size, -1, self.embed_dim))

        output = self.w0(attention_)
        return output

In [ ]:
class TransformerBlock(keras.layers.Layer):
    def __init__(self, num_heads, D, hidden_layers, dropout=0.1, norm_eps=1e-12):
        super(TransformerBlock, self).__init__()
        # Attention
        self.attention = MultiHeadAttention(
            num_heads=num_heads, embedding_dim=D,# dropout=dropout
        )
        self.norm_attention = RMSNorm(D) 
        # keras.layers.LayerNormalization(epsilon=1e-12)  #  RMSNorm(D) 

        # MLP
        self.mlp = MLP(hidden_layers, dropout)
        self.norm_mlp = RMSNorm(D)  
        # keras.layers.LayerNormalization(epsilon=1e-12)
    
    def call(self, inputs):
        
        # Feed attention
        norm_attention = self.norm_attention(inputs)
        
        # attention = self.attention(query=norm_attention, value=norm_attention)
        attention = self.attention(norm_attention)

        # Skip Connection
        attention += inputs

        # Feed MLP
        outputs = self.mlp(self.norm_mlp(attention))

        # Skip Connection
        outputs += attention

        return outputs



class TransformerEncoder(keras.layers.Layer):
    def __init__(self, num_layers, num_heads, D, mlp_dim, dropout=0.1, norm_eps=1e-12):
        super(TransformerEncoder, self).__init__()

        # Create num_layers of TransformerBlock
        self.encoder = keras.Sequential(
            [
                TransformerBlock(num_heads=num_heads,
                                 D=D,
                                 hidden_layers=[mlp_dim, D],
                                 dropout=dropout,
                                 norm_eps=norm_eps)
                for _ in range(num_layers)
            ]
        )
    def call(self, inputs, *args, **kwargs):
        # ape: (..., S, D). Example: (64, 100, 768)
        outputs = self.encoder(inputs, *args, **kwargs)
        return outputs

In [ ]:
class ViT(keras.models.Model):
    def __init__(self, num_layers=12, num_heads=12, D=768, mlp_dim=512, num_classes=10, patch_size=16, image_size=224, dropout=0.1, norm_eps=1e-12):
        super(ViT, self).__init__()
        # Data augmentation
        self.data_augmentation = keras.Sequential([
            keras.layers.Rescaling(scale=1./255),
            keras.layers.Resizing(image_size, image_size),
            keras.layers.RandomFlip("horizontal"),
            keras.layers.RandomRotation(factor=0.02),
            keras.layers.RandomZoom(
                height_factor=0.2, width_factor=0.2
            ),
        ])

        # Patch embedding
        self.embedding = PatchesEmbedding(patch_size, image_size, D)

        # Encoder with transformer
        self.encoder = TransformerEncoder(
            num_heads=num_heads,
            num_layers=num_layers,
            D=D,
            mlp_dim=mlp_dim,
            dropout=dropout,
            norm_eps=norm_eps,
        )

        # MLP head
        self.mlp_head = keras.Sequential([
            RMSNorm(D),
            # keras.layers.LayerNormalization(epsilon=norm_eps),
            keras.layers.Dense(mlp_dim),
            keras.layers.Dropout(dropout),
            keras.layers.Dense(num_classes, activation='softmax'),
        ])

        self.last_layer_norm = RMSNorm(D)

    def call(self, inputs):
        # Create augmented data
        # augmented shape: (..., image_size, image_size, c)
        augmented = self.data_augmentation(inputs)

        # Create position embedding + CLS Token
        # embedded shape: (..., S + 1, D)
        embedded = self.embedding(augmented)

        # Encode patchs with transformer
        # embedded shape: (..., S + 1, D)
        encoded = self.encoder(embedded)

        # Embedded CLS
        # embedded_cls shape: (..., D)
        embedded_cls = encoded[:, 0]
        
        # Last layer norm
        y = self.last_layer_norm(embedded_cls)
        
        # Feed MLP head
        # output shape: (..., num_classes)

        output = self.mlp_head(y)

        return output

class ViTBase(ViT):
    def __init__(self, num_classes=10, patch_size=16, image_size=224, dropout=0.1, norm_eps=1e-12):
        super().__init__(num_layers=12,
                         num_heads=12,
                         D=768,
                         mlp_dim=3072,
                         num_classes=num_classes,
                         patch_size=patch_size,
                         image_size=image_size,
                         dropout=dropout,
                         norm_eps=norm_eps)

In [ ]:
# import numpy as np
# import tensorflow as tf

# from tensorflow import keras
# from keras.losses import SparseCategoricalCrossentropy
# from keras.models import Model
# from keras import layers
# from tensorflow.data import Dataset  # Import Dataset

# # Định nghĩa lớp Args để quản lý tham số
# class Args:
#     model = 'custom'  # Bạn có thể thay đổi thành 'base' nếu cần
#     num_classes = 10
#     patch_size =  4 # 2
#     num_heads = 4 # 4
#     att_size = 64# embed 64
#     num_layer = 2
#     mlp_size = 128 # 128
#     lr = 0.001
#     weight_decay = 1e-4
#     batch_size = 32  # Giảm kích thước batch
#     epochs = 10
#     image_size = 224  # Đặt lại kích thước cho CIFAR-10
#     image_channels = 3
#     train_folder = ''  # Đường dẫn đến dữ liệu huấn luyện
#     valid_folder = ''  # Đường dẫn đến dữ liệu xác thực
#     model_folder = '.output/'

# args = Args()

# if args.train_folder != '' and args.valid_folder != '':
#     # Load train images from folder 
#     train_ds = tf.keras.preprocessing.image_dataset_from_directory(
#         args.train_folder, 
#         seed=123, 
#         image_size=(args.image_size, args.image_size), 
#         shuffle=True, 
#         batch_size=args.batch_size, 
#     )

#     val_ds = tf.keras.preprocessing.image_dataset_from_directory(
#         args.valid_folder,
#         seed=123,
#         image_size=(args.image_size, args.image_size),
#         shuffle=True,
#         batch_size=args.batch_size,
#     )
# else: 
#     print("Data folder is not set. Use CIFAR-10 dataset")
#     args.image_channels = 3 
#     args.num_classes = 10

#     # Load CIFAR-10 dataset
#     (x_train, y_train), (x_val, y_val) = tf.keras.datasets.cifar10.load_data()
#     x_train = x_train.astype(np.float32) / 255.0
#     x_val = x_val.astype(np.float32) / 255.0

#     # Tạo dataset từ CIFAR-10
#     train_ds = Dataset.from_tensor_slices((x_train, y_train))
#     train_ds = train_ds.batch(args.batch_size)

#     val_ds = Dataset.from_tensor_slices((x_val, y_val))
#     val_ds = val_ds.batch(args.batch_size)

# # Khởi tạo mô hình (giả sử VitBase và VIT đã được định nghĩa)
# if args.model == 'base':
#     model = VitBase()  # Bạn cần định nghĩa lớp VitBase
# else:
#     model = ViT(  # Bạn cần định nghĩa lớp VIT
#         num_classes=args.num_classes,
#         patch_size=args.patch_size,
#         image_size=args.image_size,
#         num_heads=args.num_heads,
#         D=args.att_size,
#         mlp_dim=args.mlp_size,
#         num_layers=args.num_layer
#     )

# # Build mô hình
# model.build(input_shape=(None, args.image_size, args.image_size, args.image_channels))

# # Tạo optimizer và compile mô hình
# optimizer = keras.optimizers.AdamW(
#     learning_rate=args.lr, weight_decay=args.weight_decay
# )
# model.compile(optimizer=optimizer, loss=SparseCategoricalCrossentropy(),
#               metrics=['accuracy'])

# # Huấn luyện mô hình
# model.fit(train_ds, epochs=args.epochs, 
#           validation_data=val_ds)

# # Lưu mô hình
# # model.save(args.model_folder)


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras.losses import SparseCategoricalCrossentropy
from keras.models import Model
from keras import layers
from tensorflow.data import Dataset

# Define Args class
class Args:
    model = 'custom'
    num_classes = 10
    patch_size = 6
    num_heads = 4
    att_size =  128# 64
    num_layer = 2 # 2 
    mlp_size = 256 # 128
    lr = 0.001
    weight_decay = 1e-4
    batch_size = 32 # 32 
    epochs = 10
    image_size = 224# Set to 32 for CIFAR-10 compatibility
    image_channels = 3
    train_folder = ''
    valid_folder = ''
    model_folder = '.output/'

args = Args()

# Load dataset
if args.train_folder and args.valid_folder:
    train_ds = tf.keras.preprocessing.image_dataset_from_directory(
        args.train_folder, seed=123, image_size=(args.image_size, args.image_size),
        shuffle=True, batch_size=args.batch_size
    )
    val_ds = tf.keras.preprocessing.image_dataset_from_directory(
        args.valid_folder, seed=123, image_size=(args.image_size, args.image_size),
        shuffle=True, batch_size=args.batch_size
    )
    # Normalize images if using image_dataset_from_directory
    train_ds = train_ds.map(lambda x, y: (tf.image.convert_image_dtype(x, tf.float32), y))
    val_ds = val_ds.map(lambda x, y: (tf.image.convert_image_dtype(x, tf.float32), y))
else:
    print("Data folder is not set. Using CIFAR-10 dataset")
    args.image_channels = 3
    args.num_classes = 10
    (x_train, y_train), (x_val, y_val) = tf.keras.datasets.cifar10.load_data()
    # x_train, x_val = x_train / 255.0, x_val / 255.0

    train_ds = Dataset.from_tensor_slices((x_train, y_train)).batch(args.batch_size)
    val_ds = Dataset.from_tensor_slices((x_val, y_val)).batch(args.batch_size)

train_ds = train_ds.prefetch(tf.data.experimental.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.experimental.AUTOTUNE)

# Define the model (assuming VitBase and ViT are defined)
# Use MirroredStrategy for distributed training
# strategy = tf.distribute.MirroredStrategy()

# with strategy.scope():
# Define the model (assuming VitBase and ViT are defined)
if args.model == 'base':
    model = VitBase()  # Define or import VitBase
else:
    model = ViT(  # Define or import ViT
        num_classes=args.num_classes,
        patch_size=args.patch_size,
        image_size=args.image_size,
        num_heads=args.num_heads,
        D=args.att_size,
        mlp_dim=args.mlp_size,
        num_layers=args.num_layer
    )

# Build model with appropriate input shape
model.build(input_shape=(None, args.image_size, args.image_size, args.image_channels))

# Initialize optimizer and compile the model
optimizer = keras.optimizers.AdamW(
    learning_rate=args.lr, weight_decay=args.weight_decay
)
model.compile(optimizer=optimizer, loss=SparseCategoricalCrossentropy(), metrics=['accuracy'])
# model.summary()
# Fit the model
model.fit(train_ds, epochs=args.epochs, validation_data=val_ds)

# # Save the model (outside the strategy scope)
# model.save(args.model_folder)
